# Paper 2 — Notebook 3 of 3
# RQ4: Do Within-Session and Post-Gap Model Rankings Diverge?

## Research question

> **RQ4.** Do the rankings of models by within-session accuracy and by post-gap accuracy
> diverge, judged against a rule pre-registered before the confirmatory run?

## Experimental objective

Re-score predictions that already exist. **No model is retrained here.** The comparison uses
rank rather than magnitude because rank is the decision-relevant quantity: a practitioner
selecting a model picks the top-ranked one.

## Why this is a separate notebook

The claim's validity rests on a sequencing property that only a separate execution unit can
preserve: **the rule must be fixed while the confirmatory seed is still unrun.** Seeds 1 and
2 are exploratory — they generated the hypothesis and have already been seen. Seed 3 is the
only out-of-sample test, and the verdict rests on seed 3 alone. Folding this into Notebook 2
would make the rule and the data available to each other.

## What this notebook replaces

`P2_Completion.ipynb`, `P2_Seed2_Confirmation.ipynb`, `P2_Seed3_Inversion_Confirmation.ipynb`
— three notebooks that patched gaps in the pipeline from outside it, one of which existed
only because a fix applied in a parallel branch never reached the branch that was run.

## Prerequisite

Notebook 2 run on **junyi** at `SEED_INDEX = 0, 1, 2`, producing
`REV_junyi/`, `REV_junyi_seed2/`, `REV_junyi_seed3/`.

## Runtime

CPU, seconds.


## RUN IDENTITY

| | |
|---|---|
| **Run ID / folder** | `RQ4__junyi` |
| **Research question** | RQ4 — within-session vs post-gap ranking divergence |
| **Dataset** | `junyi` (Junyi Academy) |
| **Reads (read-only)** | `RQ2RQ3__junyi__seed0`, `RQ2RQ3__junyi__seed1`, `RQ2RQ3__junyi__seed2` |
| **Writes** | `/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi/` only |
| **Hardware** | CPU, seconds — no retraining |

**Why this notebook reads three others.** RQ4 asks whether a ranking holds across seeds; that question does not exist inside a single run. The alternative — retraining all eight models three times here — would burn hours and produce a *second* set of AUCs for quantities the RQ2/RQ3 notebooks already report, which is precisely the duplicate-result-set problem this package exists to remove.

Isolation is enforced rather than promised: producer folders are opened read-only, the sha256 of every file read is recorded in `manifest.json`, and `finalize()` **fails the run** if any file outside this folder changed while it executed.

All three real horizons (d1, d7, d28) clear the adequacy floor here.

In [1]:
# --- locate pcdt_p2.py and p2_runtime.py (package root, one level up) -------------
import os, sys, json, time
_here = os.path.abspath("")
for _c in (_here, os.path.dirname(_here), os.path.join(_here, "..")):
    if os.path.exists(os.path.join(_c, "pcdt_p2.py")):
        sys.path.insert(0, os.path.abspath(_c)); break
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg") if os.environ.get("P2_HEADLESS") else None
import matplotlib.pyplot as plt
import pcdt_p2 as P
import p2_runtime as R
print("pcdt_p2   :", P.__file__)
print("p2_runtime:", R.__file__)
# --- AUDIT GUARD (CORRECTIONS.md) -------------------------------------------------
# Refuse to run against the pre-audit library. Without this a stale pcdt_p2.py on the
# path silently reinstates the DKVMN memory-ordering defect and every table below is
# quietly wrong again.
assert hasattr(P.Config(), "dkvmn_legacy_lag"), (
    "the pcdt_p2.py on this path is the PRE-AUDIT copy: it has no dkvmn_legacy_lag "
    "field, so DKVMN still reads its value memory before writing the current "
    "interaction. Replace pcdt_p2.py and p2_runtime.py with the corrected versions.")
assert hasattr(P, "check_alignment"), "corrected pcdt_p2.py is missing check_alignment"

pcdt_p2   : /home/hamed/Downloads/Research/P02/pcdt_p2.py
p2_runtime: /home/hamed/Downloads/Research/P02/p2_runtime.py


## 2. Imports and configuration

In [2]:
# --- locate pcdt_p2.py (package root, one level up from notebooks/) ---
import os, sys, json, time
_HERE = os.getcwd()
for _c in (_HERE, os.path.dirname(_HERE), os.path.join(_HERE, "..")):
    if os.path.exists(os.path.join(_c, "pcdt_p2.py")):
        sys.path.insert(0, os.path.abspath(_c)); break
else:
    raise FileNotFoundError(
        "pcdt_p2.py not found. Run this notebook from inside P02_clean/notebooks/ "
        "with pcdt_p2.py in the parent directory.")

import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg") if os.environ.get("P2_HEADLESS") else None
import matplotlib.pyplot as plt
import pcdt_p2 as P

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
print("pcdt_p2 loaded from:", P.__file__)
print("environment:", P.environment())


pcdt_p2 loaded from: /home/hamed/Downloads/Research/P02/pcdt_p2.py
environment: {'python': '3.13.9', 'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39', 'numpy': '2.3.5', 'pandas': '2.3.3', 'torch': '2.12.0.dev20260408+cu128', 'cuda': True, 'gpu': 'NVIDIA GeForce RTX 5080'}


In [3]:
# ============================================================================
# CONFIGURATION — GENERATED. Pinned to ONE dataset. Nothing to edit.
#
# This notebook is a CONSUMER. It opens three producer folders READ-ONLY, records the
# sha256 of every file it reads, and writes only into its own folder. finalize() fails
# the run if any file outside this folder changed while it executed.
# ============================================================================
RUN_ID   = "RQ4__junyi"
DATASET  = "junyi"
OUT_ROOT = "/home/hamed/Downloads/Research/P02/PCDT_P2_runs"

# Seed roles are fixed HERE, before any result is read.
RUNS = {"s1": dict(run_id="RQ2RQ3__junyi__seed0", seed=20260202, role="exploratory"),
        "s2": dict(run_id="RQ2RQ3__junyi__seed1", seed=20260203, role="exploratory"),
        "s3": dict(run_id="RQ2RQ3__junyi__seed2", seed=20260204, role="CONFIRMATORY")}

FOCUS_MODEL = "PCDT"
CFG = P.Config(dataset=DATASET, root="/home/hamed/Downloads/Research/dataset", out_root=OUT_ROOT)
CTX = R.RunContext(RUN_ID, OUT_ROOT, cfg=P.asdict(CFG),
                   inputs_expected=[r["run_id"] for r in RUNS.values()],
                   description="RQ4 ranking divergence — Junyi Academy, 3 seeds")
DIRS = CTX.dirs
RES  = CTX.runs_root

print("\nrun roles (fixed before any result is read):")
for k, v in RUNS.items():
    print(f"  {k}  seed={v['seed']}  {v['role']:12s}  {v['run_id']}")

RUN RQ4__junyi
RQ4 ranking divergence — Junyi Academy, 3 seeds
started   2026-08-18 16:34:30   pid 421003
writes to /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi
reads     RQ2RQ3__junyi__seed0, RQ2RQ3__junyi__seed1, RQ2RQ3__junyi__seed2

run roles (fixed before any result is read):
  s1  seed=20260202  exploratory   RQ2RQ3__junyi__seed0
  s2  seed=20260203  exploratory   RQ2RQ3__junyi__seed1
  s3  seed=20260204  CONFIRMATORY  RQ2RQ3__junyi__seed2


## 3. Load the existing per-seed results

Reads `table9_postgap_auc_by_model.csv` and `predictions_test.npz` from each seed folder.
Missing folders accumulate in an error list and are printed at the end — a missing file must
never silently become a conclusion.


In [4]:
ERRORS, T9, NPZ = [], {}, {}
for k, v in RUNS.items():
    base = CTX.sibling(v["run_id"])
    if base is None:
        ERRORS.append(f"{k}: producer run folder not found: {v['run_id']}")
        print(f"  MISSING RUN    {v['run_id']}")
        continue
    if os.path.basename(base).startswith("FIXTURE_"):
        print(f"  {k}: *** FIXTURE producer — outputs are NOT results ***")
    p9 = CTX.read_input(os.path.join(base, "tables", "table9_postgap_auc_by_model.csv"),
                        label=f"{v['run_id']}/table9")
    pz = CTX.read_input(os.path.join(base, "artifacts", "predictions_test.npz"),
                        label=f"{v['run_id']}/predictions", required=False)
    if p9:
        T9[k] = pd.read_csv(p9)
        print(f"  {k}: table9 loaded ({len(T9[k])} models)")
    else:
        ERRORS.append(f"{k}: MISSING table9 in {v['run_id']}")
    if pz:
        NPZ[k] = np.load(pz)
    else:
        ERRORS.append(f"{k}: MISSING predictions_test.npz in {v['run_id']} "
                      f"(secondary magnitude rule needs it)")

if "s3" not in T9:
    print(f"\n*** The CONFIRMATORY seed is absent for {DATASET}. No verdict can be issued. "
          f"Run 02_RQ2RQ3__{DATASET}__seed2.ipynb first. ***")
for e in ERRORS:
    print("  ERROR:", e)

  read input     RQ2RQ3__junyi__seed0/table9
  read input     RQ2RQ3__junyi__seed0/predictions
  s1: table9 loaded (8 models)
  read input     RQ2RQ3__junyi__seed1/table9
  read input     RQ2RQ3__junyi__seed1/predictions
  s2: table9 loaded (8 models)
  read input     RQ2RQ3__junyi__seed2/table9
  read input     RQ2RQ3__junyi__seed2/predictions
  s3: table9 loaded (8 models)


## 4. Rankings per seed

In [5]:
REAL = P.REAL_HORIZONS
rank_rows = []
for k in [x for x in ("s1", "s2", "s3") if x in T9]:
    t = T9[k].copy()
    adequate = [h for h in REAL if t[f"auc_{h}"].notna().any()]
    gap_leaders = {h: t.loc[t[f"auc_{h}"].idxmax(), "model"] for h in adequate}
    top_within = t.loc[t["auc_within"].idxmax(), "model"]
    for r in t.itertuples():
        rank_rows.append(dict(seed=k, role=RUNS[k]["role"], model=r.model,
                              auc_within=r.auc_within, rank_within=r.rank_within,
                              auc_mean_postgap=r.auc_mean_postgap,
                              rank_postgap=r.rank_mean_postgap))
    print(f"\n--- {k} ({RUNS[k]['role']}, seed {RUNS[k]['seed']}) ---")
    print(f"  top within session : {top_within}")
    print(f"  gap leaders        : {gap_leaders}")
    cols = ["model", "auc_within", "rank_within", "auc_mean_postgap", "rank_mean_postgap"]
    print(t.sort_values("auc_within", ascending=False)[cols].round(4).to_string(index=False, na_rep="n/a"))
RANKS = pd.DataFrame(rank_rows)



--- s1 (exploratory, seed 20260202) ---
  top within session : PCDT
  gap leaders        : {'d1': 'DKVMN', 'd7': 'DKVMN', 'd28': 'DKVMN'}
     model  auc_within  rank_within  auc_mean_postgap  rank_mean_postgap
      PCDT      0.7522          1.0            0.7246                3.0
PCDT-noPsi      0.7521          2.0            0.7270                2.0
     DKVMN      0.7484          3.0            0.7341                1.0
   Control      0.7454          4.0            0.7101                6.0
       AKT      0.7422          5.0            0.7244                4.0
     DKT-F      0.7354          6.0            0.7003                7.0
       DKT      0.7339          7.0            0.7001                8.0
      SAKT      0.7300          8.0            0.7219                5.0

--- s2 (exploratory, seed 20260203) ---
  top within session : PCDT-noPsi
  gap leaders        : {'d1': 'DKVMN', 'd7': 'DKVMN', 'd28': 'DKVMN'}
     model  auc_within  rank_within  auc_mean_postgap  rank

## 5. Primary rule

The candidate comparator is the model that leads across gaps on the **confirmatory** seed.
Naming it from s3 rather than from the exploratory seeds keeps the test out of sample.


In [6]:
VERDICT = {}
if "s3" in T9:
    t3 = T9["s3"]
    adequate3 = [h for h in REAL if t3[f"auc_{h}"].notna().any()]
    COMPARATOR = t3.loc[t3["auc_mean_postgap"].idxmax(), "model"]
    print(f"comparator named from the CONFIRMATORY seed: {COMPARATOR}")
    print(f"adequate real horizons on s3: {adequate3}\n")

    rows = []
    for k in [x for x in ("s1", "s2", "s3") if x in T9]:
        t = T9[k]
        n_models = len(t)
        within_rank = int(t.loc[t.model == COMPARATOR, "rank_within"].iloc[0])
        adeq = [h for h in REAL if t[f"auc_{h}"].notna().any()]
        leaders = {h: t.loc[t[f"auc_{h}"].idxmax(), "model"] for h in adeq}
        leg1 = within_rank > 3
        leg2 = bool(adeq) and all(v == COMPARATOR for v in leaders.values())
        rows.append(dict(seed=k, role=RUNS[k]["role"], n_models=n_models,
                         comparator=COMPARATOR, comparator_within_rank=within_rank,
                         leg1_outside_top3=leg1,
                         gap_leaders="; ".join(f"{h}:{leaders[h]}" for h in adeq),
                         leg2_first_all_gaps=leg2, divergence=bool(leg1 and leg2)))
    PRIMARY = pd.DataFrame(rows)
    print(PRIMARY.to_string(index=False))

    s3row = PRIMARY[PRIMARY.seed == "s3"].iloc[0]
    VERDICT["primary_confirmed"] = bool(s3row.divergence)
    # AUDIT FIX (CORRECTIONS.md, Finding B): the horizons the rule was actually tested on
    # were never recorded, so the cross-dataset summary had no way to state them.
    VERDICT["adequate_horizons"] = adequate3
    VERDICT["comparator"] = COMPARATOR
    VERDICT["failing_leg"] = ("none" if s3row.divergence else
                              ("leg1: comparator ranks inside the top 3 within session"
                               if not s3row.leg1_outside_top3
                               else "leg2: comparator does not lead at every adequate gap"))
    print(f"\nPRIMARY RULE on the confirmatory seed: "
          f"{'CONFIRMED' if s3row.divergence else 'NOT CONFIRMED — ' + VERDICT['failing_leg']}")
else:
    PRIMARY = pd.DataFrame()
    print("no confirmatory seed -> no primary verdict")


comparator named from the CONFIRMATORY seed: DKVMN
adequate real horizons on s3: ['d1', 'd7', 'd28']

seed         role  n_models comparator  comparator_within_rank  leg1_outside_top3                   gap_leaders  leg2_first_all_gaps  divergence
  s1  exploratory         8      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False
  s2  exploratory         8      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False
  s3 CONFIRMATORY         8      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False

PRIMARY RULE on the confirmatory seed: NOT CONFIRMED — leg1: comparator ranks inside the top 3 within session


In [7]:
# ============================================================================
# SENSITIVITY (NOT the pre-registered rule) — CORRECTIONS.md, Finding F
# ============================================================================
# The comparator above is named as argmax(auc_mean_postgap) on the CONFIRMATORY seed, and
# leg 2 ("first at every adequate gap") is then tested on that same seed. Leg 2 is
# therefore substantially favoured by construction: the comparator is defined as the model
# with the highest mean post-gap AUC on exactly the data the leg is evaluated against.
# Pre-registration fixes the rule in advance; it does not make selection and confirmation
# independent.
#
# This cell re-runs the identical rule with the comparator named from the EXPLORATORY
# seeds only. The pre-registered verdict above is unchanged and remains primary. If the
# two disagree, the finding is selection-dependent and must be reported as such.
SENS = {}
if "s3" in T9 and all(k in T9 for k in ("s1", "s2")):
    _mean = (pd.concat([T9["s1"][["model", "auc_mean_postgap"]],
                        T9["s2"][["model", "auc_mean_postgap"]]])
             .groupby("model")["auc_mean_postgap"].mean())
    if _mean.notna().any():
        COMP_EXPL = _mean.idxmax()
        t = T9["s3"]
        wr = int(t.loc[t.model == COMP_EXPL, "rank_within"].iloc[0])
        adq = [h for h in REAL if t[f"auc_{h}"].notna().any()]
        ldrs = {h: t.loc[t[f"auc_{h}"].idxmax(), "model"] for h in adq}
        l1, l2 = wr > 3, (bool(adq) and all(v == COMP_EXPL for v in ldrs.values()))
        SENS = dict(comparator_from_exploratory=COMP_EXPL,
                    comparator_from_confirmatory=VERDICT.get("comparator"),
                    same_comparator=bool(COMP_EXPL == VERDICT.get("comparator")),
                    within_rank_on_s3=wr, leg1=l1, leg2=l2, divergence=bool(l1 and l2))
        print(f"comparator named from exploratory seeds : {COMP_EXPL}")
        print(f"comparator named from confirmatory seed : {VERDICT.get('comparator')}")
        print(f"rule on s3 with the exploratory comparator: "
              f"{'CONFIRMED' if SENS['divergence'] else 'NOT CONFIRMED'} "
              f"(leg1={l1}, leg2={l2}, within rank={wr})")
        if SENS["divergence"] != bool(VERDICT.get("primary_confirmed")):
            print("*** SELECTION-DEPENDENT: the verdict changes with how the comparator "
                  "is named. Report both and do not present the pre-registered verdict "
                  "alone. ***")
        elif not SENS["same_comparator"]:
            print("Verdict agrees, but the two procedures name different comparators — "
                  "state which model the claim is about.")
        else:
            print("Comparator and verdict both agree: the finding is not an artefact of "
                  "naming the comparator on the seed it is tested on.")
    VERDICT["selection_sensitivity"] = SENS
else:
    print("sensitivity not evaluable (needs all three seeds)")

comparator named from exploratory seeds : DKVMN
comparator named from confirmatory seed : DKVMN
rule on s3 with the exploratory comparator: NOT CONFIRMED (leg1=False, leg2=True, within rank=3)
Comparator and verdict both agree: the finding is not an artefact of naming the comparator on the seed it is tested on.


## 6. Secondary rule — magnitude and sign flip

A rank ordering can change on a hair's breadth. The secondary rule asks whether the *signed*
difference actually flips: positive within session, negative across the gap, with bootstrap
CIs excluding zero.


In [8]:
def signed_delta(z, a, b, bucket_name, cfg):
    """Bootstrap AUC(a) - AUC(b) inside one gap bucket, resampling learners."""
    from sklearn.metrics import roc_auc_score
    bid = [i for i, (k, _, _) in enumerate(P.BUCKETS) if k == bucket_name][0]
    m, bk, y = z["mask"] > 0, z["bucket"], z["targets"]
    sel = (bk == bid) & m
    if sel.sum() < cfg.min_points_bucket:
        return None
    rows_idx = np.repeat(np.arange(z[a].shape[0])[:, None], z[a].shape[1], 1)[sel]
    pa, pb, yy = z[a][sel], z[b][sel], y[sel]
    if len(np.unique(yy)) < 2:
        return None
    d, lo, hi, pv = P.paired_bootstrap(pa, pb, yy, rows_idx, cfg.n_bootstrap, cfg.seed)
    return dict(bucket=bucket_name, n=int(sel.sum()), delta=d, ci_lo=lo, ci_hi=hi, p=pv,
                sig=bool(lo > 0 or hi < 0))

SECOND = []
if "s3" in T9 and NPZ:
    for k in [x for x in ("s1", "s2", "s3") if x in NPZ]:
        z = NPZ[k]
        if FOCUS_MODEL not in z.files or VERDICT.get("comparator") not in z.files:
            ERRORS.append(f"{k}: predictions missing {FOCUS_MODEL} or {VERDICT.get('comparator')}")
            continue
        for h in ["within"] + REAL:
            r = signed_delta(z, FOCUS_MODEL, VERDICT["comparator"], h, CFG)
            if r:
                r.update(seed=k, role=RUNS[k]["role"],
                         comparison=f"{FOCUS_MODEL} - {VERDICT['comparator']}")
                SECOND.append(r)
    SECOND = pd.DataFrame(SECOND)
    if len(SECOND):
        print(SECOND[["seed", "role", "comparison", "bucket", "n", "delta",
                      "ci_lo", "ci_hi", "sig"]].round(4).to_string(index=False))
        flips = {}
        for k in SECOND.seed.unique():
            s = SECOND[SECOND.seed == k]
            w = s[s.bucket == "within"]
            g = s[s.bucket.isin(REAL)]
            pos = bool(len(w) and w.iloc[0].delta > 0 and w.iloc[0].sig)
            neg = bool(((g.delta < 0) & g.sig).any())
            flips[k] = dict(within_positive_sig=pos, any_gap_negative_sig=neg,
                            sign_flip=bool(pos and neg))
        print("\nsign flip by seed:", json.dumps(flips, indent=2))
        VERDICT["secondary_confirmed"] = bool(flips.get("s3", {}).get("sign_flip", False))
        VERDICT["sign_flip_by_seed"] = flips
else:
    SECOND = pd.DataFrame()
    print("secondary rule not evaluated (needs the confirmatory seed and its predictions)")


seed         role   comparison bucket      n   delta   ci_lo   ci_hi   sig
  s1  exploratory PCDT - DKVMN within 673602  0.0038  0.0029  0.0047  True
  s1  exploratory PCDT - DKVMN     d1  15132 -0.0026 -0.0070  0.0019 False
  s1  exploratory PCDT - DKVMN     d7   6490 -0.0133 -0.0203 -0.0065  True
  s1  exploratory PCDT - DKVMN    d28   3171 -0.0127 -0.0235 -0.0025  True
  s2  exploratory PCDT - DKVMN within 676908  0.0026  0.0017  0.0035  True
  s2  exploratory PCDT - DKVMN     d1  15063 -0.0060 -0.0106 -0.0012  True
  s2  exploratory PCDT - DKVMN     d7   6625 -0.0089 -0.0167 -0.0018  True
  s2  exploratory PCDT - DKVMN    d28   3100 -0.0135 -0.0243 -0.0014  True
  s3 CONFIRMATORY PCDT - DKVMN within 669904  0.0027  0.0019  0.0036  True
  s3 CONFIRMATORY PCDT - DKVMN     d1  15025 -0.0104 -0.0148 -0.0061  True
  s3 CONFIRMATORY PCDT - DKVMN     d7   6885 -0.0073 -0.0139 -0.0007  True
  s3 CONFIRMATORY PCDT - DKVMN    d28   3251 -0.0087 -0.0195  0.0015 False

sign flip by seed: {
  "

## 7. Verdict, with the qualifications attached

The qualifications are printed with the verdict rather than left for a reader to derive,
because a robust *ordering* effect with small *magnitudes* is easy to overstate.


In [9]:
if VERDICT.get("primary_confirmed") is not None:
    ok = VERDICT["primary_confirmed"]
    print("=" * 78)
    print(f"RQ4 VERDICT: {'CONFIRMED' if ok else 'NOT CONFIRMED'} on the out-of-sample "
          f"seed {RUNS['s3']['seed']}")
    print("=" * 78)
    print(f"  primary rule   : {'passed' if ok else 'failed — ' + VERDICT['failing_leg']}")
    print(f"  secondary rule : {'passed' if VERDICT.get('secondary_confirmed') else 'not passed'}")
    print(f"  comparator     : {VERDICT['comparator']}")

    if len(SECOND):
        w = SECOND[(SECOND.bucket == "within")]
        big = (w.delta.abs() >= CFG.min_meaningful_auc_delta).any()
        print("\n  QUALIFICATIONS (report these with the finding):")
        print(f"   1. Within-session |delta| vs the MMDE ({CFG.min_meaningful_auc_delta}): "
              f"{'at or above' if big else 'BELOW'}. "
              + ("" if big else "So this is a MODEL-SELECTION result, not a claim that one "
                                "model is meaningfully better within session."))
        weak = SECOND[(SECOND.bucket == "d28") & (~SECOND.sig)]
        if len(weak):
            print(f"   2. The 4-week leg is the weakest: CI includes zero on "
                  f"{sorted(set(weak.seed))}. Claim confirmation at 1-day and 1-week; report "
                  f"4 weeks as directional only.")
        print("   3. Demonstrated on one dataset. Say so.")
        print("   4. Disclose that the FIRST pre-registered rule failed and was re-specified "
              "BEFORE the confirmatory seed was run.")

    print("\n  A result that does not depend on the authors' own model is worth reporting: "
          "if any pair of PUBLISHED baselines shows the same reversal, the point stands "
          "without reference to the proposed representation at all.")
    if "s3" in T9:
        t3 = T9["s3"].dropna(subset=["auc_mean_postgap"])
        base = t3[t3.model.isin(P.BASELINES + ["Control"])]
        if len(base) >= 2:
            bw = base.sort_values("auc_within", ascending=False).iloc[0]["model"]
            bg = base.sort_values("auc_mean_postgap", ascending=False).iloc[0]["model"]
            print(f"     among baselines + Control only: best within = {bw}, "
                  f"best across gaps = {bg}"
                  + ("  <- same reversal, no PCDT involved" if bw != bg else ""))

for e in ERRORS:
    print("  ERROR:", e)


RQ4 VERDICT: NOT CONFIRMED on the out-of-sample seed 20260204
  primary rule   : failed — leg1: comparator ranks inside the top 3 within session
  secondary rule : passed
  comparator     : DKVMN

  QUALIFICATIONS (report these with the finding):
   1. Within-session |delta| vs the MMDE (0.01): BELOW. So this is a MODEL-SELECTION result, not a claim that one model is meaningfully better within session.
   2. The 4-week leg is the weakest: CI includes zero on ['s3']. Claim confirmation at 1-day and 1-week; report 4 weeks as directional only.
   3. Demonstrated on one dataset. Say so.
   4. Disclose that the FIRST pre-registered rule failed and was re-specified BEFORE the confirmatory seed was run.

  A result that does not depend on the authors' own model is worth reporting: if any pair of PUBLISHED baselines shows the same reversal, the point stands without reference to the proposed representation at all.
     among baselines + Control only: best within = DKVMN, best across gaps = DKVM

## 8. Save artefacts

In [10]:
tabs = {"table10_rankings_by_seed": RANKS}
if SENS:
    tabs["table13_selection_sensitivity"] = pd.DataFrame([SENS])
if len(PRIMARY):
    tabs["table11_primary_rule_by_seed"] = PRIMARY
if len(SECOND):
    tabs["table12_secondary_magnitude"] = SECOND
P.save_tables(tabs, DIRS["tab"])

rec = dict(notebook="03_RQ4_ranking_divergence_confirmation",
           generated=time.strftime("%Y-%m-%d %H:%M:%S"), dataset=DATASET,
           runs={k: v for k, v in RUNS.items()},
           preregistered_primary_rule=("comparator ranks outside the top 3 within session AND "
                                       "first at every adequate real gap horizon, on the "
                                       "confirmatory seed"),
           preregistered_secondary_rule=("bootstrap CI for AUC(PCDT)-AUC(comparator) positive "
                                         "within session and negative at >=1 adequate real gap"),
           verdict=VERDICT, errors=ERRORS, environment=P.environment())
with open(os.path.join(DIRS["art"], "verdict_rq4.json"), "w") as fh:
    json.dump(rec, fh, indent=2, default=str)
print(f"wrote {DIRS['out']}/verdict_rq4.json")


tables -> /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi/tables: ['table10_rankings_by_seed', 'table11_primary_rule_by_seed', 'table12_secondary_magnitude', 'table13_selection_sensitivity']
wrote /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi/verdict_rq4.json


## 9. Conclusion — what RQ4 supports

If the rule passes on the out-of-sample seed, the supported claim is:

> On this dataset, the model selected by the standard within-session benchmark is **not** the
> model that performs best across real inter-session gaps. Model *selection* under the field's
> default protocol can therefore be wrong for longitudinal deployment.

What it does **not** support: that any particular model is meaningfully better; that the
effect generalises to the other datasets; that the four-week horizon is as well evidenced as
the one-day and one-week horizons.

The mechanism needs nothing exotic. Ranking divergence follows from models degrading across
gaps at **different rates** — which the within-vs-post-gap scatter in Notebook 2 shows
directly.


In [11]:
CTX.finalize(
    status=("complete" if VERDICT.get("primary_confirmed") is not None
            else "incomplete — no verdict issued"),
    notes="; ".join(ERRORS) if ERRORS else "",
    extra=dict(research_question="RQ4", dataset=DATASET,
               seed_roles={k: v["role"] for k, v in RUNS.items()},
               verdict=VERDICT))


RUN RQ4__junyi — complete
  runtime        15.92 min
  files written  10
  inputs read    6 (0 missing)
  isolation      OK — no finished run was modified
  manifest       /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi/manifest.json


{'run_id': 'RQ4__junyi',
 'description': 'RQ4 ranking divergence — Junyi Academy, 3 seeds',
 'status': 'complete',
 'notes': '',
 'fixture': False,
 'started': '2026-08-18 16:34:30',
 'finished': '2026-08-18 16:50:25',
 'runtime_minutes': 15.92,
 'output_dir': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4__junyi',
 'environment': {'python': '3.13.9',
  'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'torch': '2.12.0.dev20260408+cu128',
  'sklearn': '1.7.2',
  'scipy': '1.18.0',
  'matplotlib': '3.10.6',
  'cuda': True,
  'gpu': 'NVIDIA GeForce RTX 5080'},
 'config': {'dataset': 'junyi',
  'root': '/home/hamed/Downloads/Research/dataset',
  'out_root': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs',
  'max_learners': 20000,
  'min_seq_len': 10,
  'max_seq_len': 200,
  'train_frac': 0.5,
  'val_frac': 0.1,
  'test_frac': 0.4,
  'split_mode': 'learner',
  'param_budget': 350000,
  'param_tolerance': 0.05,
  'batch_size'